In [1]:
import json
import os
import re

def read_txt_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    return content

def parse_criteria(criteria_lines):
    criteria_tree = []
    for line in criteria_lines:
        line = line.strip()
        if "[AND]" in line or "[OR]" in line:
            parts = re.split(r'\s*\[\s*(AND|OR)\s*\]\s*', line)
            for part in parts:
                if part.strip():
                    print("part", part.strip())
                    criteria_tree.append(part.strip())
        else:
            criteria_tree.append(line)
    return criteria_tree

def build_tree_with_conjunctions(criteria_tree, outer_conjunction="AND"):
    if not criteria_tree:
        return None

    def split_criteria(criteria_list):
        result = []
        temp_list = []
        for criteria in criteria_list:
            if criteria.upper() in ["AND", "OR"]:
                if temp_list:
                    result.append(temp_list)
                    temp_list = []
                result.append(criteria.upper())
            else:
                temp_list.append(criteria)
        if temp_list:
            result.append(temp_list)
        return result

    def recursive_build(criteria_list):
        print("building tree with", criteria_list)
        if not criteria_list:
            return None
        if len(criteria_list) == 1 and isinstance(criteria_list[0], list):
            return {"raw_text": criteria_list[0][0]} if len(criteria_list[0]) == 1 else build_tree_with_conjunctions(criteria_list[0], "AND")
        if len(criteria_list) == 1:
            return {"raw_text": criteria_list[0]}

        left = recursive_build(criteria_list[:-2])
        right = {"raw_text": criteria_list[-1][0]} if len(criteria_list[-1]) == 1 else build_tree_with_conjunctions(criteria_list[-1], "AND")

        return {criteria_list[-2]: {"left": left, "right": right}}

    split_list = split_criteria(criteria_tree)
    return recursive_build(split_list)

def create_tree_structure(text):
    lines = text.split('\n')
    inclusion_criteria = []
    exclusion_criteria = []
    print("lines", lines)
    current_list = None
    for line in lines:
        line = line.strip()
        if line.startswith("Inclusion Criteria"):
            current_list = inclusion_criteria
        elif line.startswith("Exclusion Criteria"):
            current_list = exclusion_criteria
        elif line.startswith('-') and current_list is not None:
            current_list.append(line[1:].strip())

    inclusion_tree = build_tree_with_conjunctions(parse_criteria(inclusion_criteria), "AND")
    exclusion_tree = build_tree_with_conjunctions(parse_criteria(exclusion_criteria), "OR")

    return {"AND": {"left": inclusion_tree, "right": {"NOT OR": exclusion_tree}}}

def save_json(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4)

In [ ]:
file_path = 'data_parsed_1/NCT03864393.txt'
output_path = 'data_parsed_2/NCT03864393.json'

text = read_txt_file(file_path)
tree_structure = create_tree_structure(text)
save_json(tree_structure, output_path)
